# 06 - Technical & Options Positioning Signals

**Author:** Sacha Huberty

**Purpose:** Build the per-asset technical/positioning signal (pivot
detection, K-Means support/resistance zones, an event-study sanity
check), plus a live-only options-positioning snapshot (OI notional,
call/put walls, gamma proxy -- yfinance has no option-chain history,
so these cannot be backtested). Wire the price-level component in as
V3 plus execution timing (phased entries near resistance), and
backtest the resulting strategy OOS.

**Last updated:** 2026-07-25

## Setup

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import (
    allocation, backtest, data, metrics, regimes, strategy,
    strategy_legacy, technicals, universe,
)

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["technicals"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)
returns.tail()

## Analysis / signal logic

### Pivots and K-Means support/resistance zones (in-sample snapshot)

One representative asset, chosen for having the tightest, most
distinct zones (a good visual illustration), plotted with its pivot
highs/lows and the resulting K-Means zone levels.

In [ ]:
in_sample_prices = prices.loc[:as_of_universe]
diag = technicals.technical_signal(in_sample_prices, cfg)
diag.sort_values("distance_bps", key=lambda s: s.abs())

In [ ]:
example_ticker = "SPY" if "SPY" in tickers else tickers[0]
series = in_sample_prices[example_ticker].dropna()
pivots = technicals.find_pivots(series, cfg["technicals"]["pivot_order"])
zones = technicals.sr_zones(series, cfg)

fig, ax = plt.subplots(figsize=(12, 5))
tail = series.tail(500)
ax.plot(tail.index, tail.values, linewidth=0.8, color="black")
highs = pivots["pivot_high"].reindex(tail.index).fillna(False)
lows = pivots["pivot_low"].reindex(tail.index).fillna(False)
ax.scatter(tail.index[highs], tail[highs], color="red", s=15, label="pivot high", zorder=3)
ax.scatter(tail.index[lows], tail[lows], color="green", s=15, label="pivot low", zorder=3)
for zone in zones:
    ax.axhline(zone, color="blue", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title(f"{example_ticker}: pivots and K-Means S/R zones")
ax.legend()
plt.tight_layout()
plt.show()

### Event study: does price actually bounce/reject at these zones?

In [ ]:
event = technicals.event_study(in_sample_prices, cfg, horizon_days=10)
event.sort_values("hit_rate", ascending=False)

### Live options positioning (NOT backtested)

yfinance only provides the CURRENT option chain -- there is no
history to backtest against, per PROJECT_STRUCTURE.md section 7. This
section is a live snapshot only, for the forward paper-trading phase,
not for the OOS backtest below.

In [ ]:
optionable_ticker = "SPY" if "SPY" in tickers else tickers[0]
try:
    chain = data.download_option_chain(optionable_ticker)
    spot = float(prices[optionable_ticker].iloc[-1])
    walls = technicals.call_put_walls(chain)
    notional = technicals.oi_notional(chain, spot)
    proxy = technicals.gamma_proxy(chain, spot)
    print(f"{optionable_ticker} spot: {spot:.2f}")
    print(f"Call wall: {walls['call_wall']}, Put wall: {walls['put_wall']}")
    print(f"OI notional: ${notional:,.0f}")
    print(f"Gamma proxy (near-spot call/put OI imbalance): {proxy:.3f}")
except Exception as exc:
    print(f"Live option chain unavailable right now: {exc}")

### V3 + execution timing -> backtest OOS

Layers `with_technical_view` on top of stage 5's full pipeline
(V1 + anomaly + mean-reversion), and passes `technical_phase_flags`
as backtest.run's `phase_flags_fn`: any rebalance touching a
resistance-flagged asset has its WHOLE trade scaled down this week
(not just that asset's leg -- scaling one leg alone would break the
sum-to-1 weight invariant), catching up over subsequent rebalances.
Buffered by `optimization.lookback_days` (not the longer HMM lookback
stages 4-5 used) to keep this stage's added per-week cost (KMeans
S/R zones on top of the HMM, autoencoder, and mean-reversion signals)
tractable -- safe now that `regimes.has_enough_history` and
`technicals`/`anomaly`'s equivalents guard the shorter cold start.

In [ ]:
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
opt_lookback = cfg["optimization"]["lookback_days"]

buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - opt_lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

backtest_cfg = copy.deepcopy(cfg)
backtest_cfg["anomaly"]["epochs"] = 10
backtest_cfg["anomaly"]["patience"] = 3
backtest_cfg["anomaly"]["refit_frequency_days"] = 126

v1_fn = strategy_legacy.regime_switching_strategy(class_bucket, backtest_cfg, posture_cfg)
v1_anomaly_fn = strategy.with_anomaly_override(v1_fn, backtest_cfg)
v1_anomaly_meanrev_fn = strategy_legacy.with_meanreversion_tilt(v1_anomaly_fn, backtest_cfg)
full_fn = strategy_legacy.with_technical_view(v1_anomaly_meanrev_fn, backtest_cfg)
phase_flags_fn = strategy_legacy.technical_phase_flags(backtest_cfg)

full_result_bt = backtest.run(
    full_fn, backtest_returns, backtest_cfg, phase_flags_fn=phase_flags_fn
)

In [ ]:
# Stage 2-5 baselines, recomputed here (same buffered range) for a
# direct comparison.
lookback = cfg["optimization"]["lookback_days"]


def make_classical_strategy(method):
    cov_method = cfg["optimization"]["covariance"]

    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


v1_only_fn = strategy_legacy.regime_switching_strategy(class_bucket, cfg, posture_cfg)
v1_anomaly_meanrev_baseline_fn = strategy_legacy.with_meanreversion_tilt(
    strategy.with_anomaly_override(v1_only_fn, backtest_cfg), backtest_cfg
)

baseline_fns = {
    "permanent": permanent_strategy,
    "risk_parity": make_classical_strategy("risk_parity"),
    "regime_switching_anomaly_meanrev": v1_anomaly_meanrev_baseline_fn,
}
baseline_results = {
    name: backtest.run(fn, backtest_returns, cfg)
    for name, fn in baseline_fns.items()
}
all_results = {"full_v1_v2_v3": full_result_bt, **baseline_results}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison.sort_values("sharpe", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in all_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "full_v1_v2_v3" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: V1 + anomaly + V2 + V3 vs. baselines")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend()
plt.show()

## Notes / next steps

- **Honest findings from this run:** the event study shows near-cash instruments (BIL, SHY) generate huge numbers of "events" (2533 and 1301 respectively) with hit rates BELOW chance (31.6% and 45.9%) -- their near-flat price behavior triggers frequent spurious "near a zone" flags, not genuine support/resistance structure. K-Means S/R zones (and this event study) are much more meaningful for assets with real price variation. OOS, the full pipeline (V1+anomaly+meanrev+technicals, Sharpe 0.738) came in marginally BELOW the prior stage's combo alone (0.7487) -- consistent with the ongoing honest pattern that each naive additive view hasn't yet clearly improved on the last, pending Black-Litterman's proper fusion in stage 8. Note the exact Sharpe figures here aren't perfectly apples-to-apples with notebook 05's own numbers for the same combo, since this notebook buffers by the shorter `optimization.lookback_days` rather than the HMM's longer lookback (a stage-6 scope trade for runtime, not a change in the strategy logic) -- the relative conclusion holds regardless.

- Options positioning (OI notional, call/put walls, gamma proxy) is a
  live-only snapshot in this notebook, never part of the OOS backtest
  -- yfinance has no historical option chains, so there is nothing to
  backtest it against. It's evaluated forward, in paper trading.
- Execution timing phases the WHOLE trade, not just the flagged
  asset's leg: scaling one leg alone breaks the sum-to-1 weight
  invariant (if one asset's entry is held back, another's exit must be
  held back correspondingly). Confirmed with an exact geometric-
  convergence test while building backtest.py's `phase_flags_fn`.
- Per PROJECT_STRUCTURE.md's overfitting defense, whether V3 earns its
  complexity is reported honestly via the comparison table above; the
  formal per-module marginal-Sharpe verdict is the ablation study in
  notebook 11.
- Next (stage 7): `sentiment.py` (scraping, NLP, per-asset-class
  sentiment score, lagged one day), added as V4 -- the last naive view
  before Black-Litterman (stage 8) formalizes the fusion.